In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.." 
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}

spike_cifar = pd.read_csv(f"{REPO}/results/20260816_230437_38678/csv/spike_selection.csv")
spike_imagenet = pd.read_csv(f"{REPO}/results/20260816_083054_38677/csv/spike_selection.csv")
spike_all = pd.concat([spike_cifar, spike_imagenet], ignore_index=True)
spike_all

,model,dataset,spike_by_trace,spike_by_damage,agreement
0,resnet50_no_weights,CIFAR10,conv1,fc,no
1,resnet18_no_weights,CIFAR10,layer1.1.conv1,layer1.1.conv1,yes
2,cnn,CIFAR10,conv2,conv1,no
3,resnet50_no_weights,IMAGENET100,conv1,conv1,yes
4,resnet18_no_weights,IMAGENET100,conv1,conv1,yes
5,cnn,IMAGENET100,conv3,conv1,no


In [ ]:
MODELS = ["cnn", "resnet18_no_weights", "resnet50_no_weights"]
cifar = spike_cifar.set_index("model").loc[MODELS].reset_index()
table7 = pd.DataFrame({
    "Modell": [MODEL_LABEL[m] for m in MODELS],
    "Trace-Spike": cifar["spike_by_trace"].values,
    "Damage-Spike": cifar["spike_by_damage"].values,
})
table7["Übereinstimmung"] = np.where(cifar["agreement"].values == "yes", "ja", "nein")
table7.to_csv(f"{FIG_DIR}/tab_07_spike_selection.csv", index=False)
table7

,Modell,Trace-Spike,Damage-Spike,Übereinstimmung
0,CNN,conv2,conv1,nein
1,ResNet-18,layer1.1.conv1,layer1.1.conv1,ja
2,ResNet-50,conv1,fc,nein


In [ ]:
canon = pd.read_csv(f"{REPO}/results/20260816_230437_38678/csv/canonical_traces.csv")
ranks = pd.read_csv(f"{REPO}/results/review_response/csv/normalized_ranks_loss.csv")

def trace_spike(model):
    sub = canon[(canon["model"] == model) & (canon["dataset"] == "CIFAR10")
                & (canon["variant"] == "fp32_fused") & (canon["seed"] == "aggregate")]
    return sub.loc[sub["trace_raw"].idxmax(), "canonical_layer"]

damage_spike = (ranks[(ranks["dataset"] == "CIFAR10") & (ranks["predictor"] == "raw_trh")]
                 .set_index("model")["true_top_layer"].to_dict())

for m in MODELS:
    assert trace_spike(m) == cifar.set_index("model").loc[m, "spike_by_trace"], m
    assert damage_spike[m] == cifar.set_index("model").loc[m, "spike_by_damage"], m
print("Cross-check OK: independent derivation matches spike_selection.csv for all 3 models.")

Cross-check OK: independent derivation matches spike_selection.csv for all 3 models.


In [ ]:
random_init = pd.read_csv(f"{REPO}/results/20260816_230437_38678/csv/random_init_summary.csv")
random_init_default = random_init[random_init["bn_mode"] == "default"]
random_init_default[["model", "dataset", "spike_layer", "verdict"]]

,model,dataset,spike_layer,verdict
0,resnet50_no_weights,CIFAR10,layer4.1.conv2.weight,architectural
1,resnet18_no_weights,CIFAR10,layer3.0.conv2.weight,architectural
2,cnn,CIFAR10,conv3.weight,learned
